# Hierarchical Level of Detail Modelling

In this notebook we explore the basics of HLOD modelling and how best to structure our glTF files so that they may be loaded seamlessly into a large scene. The test model we will be working with is of the Sixty5 model- specifically the MEP layer.

## Imports

In [2]:
import numpy as np
import pygltflib
import pandas as pd

## Introduction

HLOD modeling is prevalent in the GIS space, especially in applications such as Google Earth. Here, the vast amount of data that needs to be stored and loaded when the user opens the application, makes this necessary. Here, only the specific data near the user is loaded to the scene, while everything else is unloaded to save memory. As a result, entire cities can be quickly loaded even on devices with minimal compute.

Here, we will attempt to recreate the process using processes and tools we've seen before with a large BIM scene.

The goal will be to organize our data into spatial cells, which will store the individual geometries of our scene. As the camera moves around the scene, the different cells are dynamically loaded and unloaded from the screen, in theory- capping the total amount of memory being used by the scene, while still providing a seamless user experience. Let's jump in.

## Octrees and glTF

It may be useful to work our way backwards here, i.e. starting from the three.js render cycle. In my head, at it simplest, the render cycle should consist of 2 passes. One pass will traverse our octree structure and determine the closest objects to the camera, and add those items to a `set()`. Another function will iterate over the items in the set and see if they are still close to the camer (i.e. if the camera has moved, and the objects no longer need to be rendered). The renderer will send draw calls for each item in the render set.

Now, the draw calls are handled in the function `renderer.render( scene, camera )`. This means, the objects will need to be within our scene graph in order to be caught in the draw call. In theory, this would mean calling `scene.add( child )` for each mesh in the traversal step. As well, this would mean that objects in the `set()` will need to have `scene.remove( object )` called. Per notes on three.js, we will also need to call `.dispose()` on the geometry to truly remove it from memory.

In [3]:
# Scene
# - node 1
#   children: {
#     9, 10, 11, 12, 13, 14, 15, 16
#   },
#   bounds: {
#   
#   },
#   meshes: {
#   
#   }
# - node 2
#   children: {
#     
#   },
#   bounds: {
#   
#   },
#   meshes: {
#   
#   }
# ...
# 
# - node 8
#   children: {
#   
#   },
#   bounds: {
#   
#   },
#   meshes: {
#   
#   }

Every node will have 3 attributes- children, bounds, meshes. The children refer to the octree sub nodes for a given node, and there should be 8 of them. The bounds refers to the bounding volume of the node, and the meshes refer to any meshes that are stored in that level. In my head, meshes that occupy a large space in the scene (i.e. have a large bounding volume), should be stored higher up the tree. Hence, if we have something like a large concrete slab, this object should sit in the higher nodes, that way when we test to see if our camera intersects with the node, we will draw the slab to the screen, even if we're far away.

Another note, we will need a base layer active at all times such that the user will know where to navigate around. Theoretically, this will need to be stored in the root node, and this will always be active.

So, continuing on- how do we create this octree structure. Its obvious that we'll need the bounding boxes of each object in the scene to start with. These can be calculated on the fly if needed. Whats less obvious is where to store them. As mentioned earlier, larger objects should be stored higher up the tree. When these objects become visible in the scene will be our next critical thinking hurdle.

An object that is completely enclosed within the bounds of a node, will belong to that node. An object whose bounds intersect more than one node should be tested to see if it belongs in the parent. If it is completely enclosed within the parent, it will belong to that node. If not, continue up. This is a bottom up approach that may prove to be a little computationally expensive however, it will only need to be done once. Let's consider this the naive approach.

Let's start by testing this approach out with a sample of objects. Here is what they look like.

![HLOD Sample Objects](../img/HLOD-test-objects.png)

In [4]:
gltf_sample = pygltflib.GLTF2().load('data/sixty5-w-installatie-sample-objects.glb')

In [5]:
len(gltf_sample.nodes)

31

We have 31 objects, with a healthy mix of sizes.

Onto the bounding boxes, how might we best calculate this? Let's see if there's any info in the meshes that we can use to calculate this. We need to start with the bounding box of the scene.

In [6]:
for mesh in gltf_sample.meshes:
    print(mesh.name)
    
    # accessor = mesh.primitives[0].attributes.POSITION
    # print(gltf_sample.accessors[accessor].max)

102/2060837
102/2059624
102/2276949
102/2283033
102/2075528
102/2079477
102/2080746
102/1457679
102/2081942
102/2079846
102/2288413
102/2299202
102/1299604
102/115261
102/2281639
102/1937005
102/1240813
102/2075528.001
102/1299604.001
102/1937005.001
102/2283033.001
102/1299604.002
102/1299604.003
102/1299604.004
102/1299604.005
102/1299604.006
102/1299604.007
102/2080746.001
102/2080746.002
102/2080746.003
102/2080746.004


We may be able to use the data in the max and min columns. Lets see.

In [7]:
min_max_dict = {}

for mesh in gltf_sample.meshes:
    accessor_idx = mesh.primitives[0].attributes.POSITION
    accessor = gltf_sample.accessors[accessor_idx]
    
    min_max_dict[mesh.name] = [
        accessor.max[0], accessor.max[1], accessor.max[2],
        accessor.min[0], accessor.min[1], accessor.min[2],
    ]

In [8]:
df = pd.DataFrame.from_dict(min_max_dict, orient='index', columns=["max_x", "max_y", "max_z", "min_x", "min_y", "min_z"])
df.head()

,max_x,max_y,max_z,min_x,min_y,min_z
102/2060837,0.637176,0.040000,0.039768,-0.637136,-0.040000,-0.040230
102/2059624,0.096035,0.042975,1.914585,-0.098965,-0.037025,-1.887310
102/2276949,0.087498,0.149999,1.576775,-0.087502,-0.150001,-1.576775
102/2283033,0.600000,0.087501,0.078434,-0.600000,-0.087499,-0.078449
102/2075528,0.058229,0.041000,0.062500,-0.066771,-0.041000,-0.062500


Now, we must find the global min and max, such that we can create the scene node's bounds.

In [9]:
np.max(df, axis=0)

max_x    0.793247
max_y    1.587500
max_z    3.906563
min_x   -0.013666
min_y   -0.037025
min_z   -0.014023
dtype: float64

As well as the min.

In [10]:
np.min(df, axis=0)

max_x    0.014334
max_y    0.038206
max_z    0.013977
min_x   -0.793253
min_y   -1.587500
min_z   -4.427464
dtype: float64

In [11]:
np.max([np.abs(np.max(df, axis=0)), np.abs(np.min(df, axis=0))], axis=0)

array([0.79324722, 1.58749998, 3.90656281, 0.79325294, 1.58749998,
       4.42746353])

Since our first node must be a cube, we need to take the absolute max of each dimension and use that as the bounds. Here, our largest dimension is z whose bounds extend out 3.9m in the positive axis and 4.42m in the negative axis. To account for this, we double our largest dimension- 4.427 which equals 8.854, to center our cube while still accounting for everything. This will the dimensions of our input cube.

At this point it may be good to pause and see what our plan is here. I was thinking the best way to do this would be to start at a root cube and make our way down the tree by recursively subdividing. I've heard of different approachees which start with a bottom up method, i.e. we start with the smallest objects and make our way up. I'm not sure which method may be best for this approach, but lets start with the top down.

As well, our glTF file should be able to store the information needed to accurately traverse the scene. This means each node must contain its bounds as well as a reference to its children. Hence Node 1 (corresponding to the scene node) must contain at least the following information.

In [ ]:
# Scene
# - node 1
#   children: {
#     2,3,4,5,6,7,8,9
#   },
#   bounds: {
#      min_x : -4.427
#      min_y : -4.427
#      min_z : -4.427
#      extents : 8.854
#   },
#   meshes: {
#      ...
#   }

We will acquire the required precision later, and focus just on the structure of this object for now.

Moving down to the children, each child will have half the extents of its parent. We will need to write a recursive function to acquire this information and save it to the object. For example, node 2 and 3 could have the following spatial data.

In [ ]:
# Scene
# - node 1
#       ...
# - node 2
#   children: {
#     10,11,12,13,14,15,16,17
#   },
#   bounds: {
#      min_x : -4.427
#      min_y : -4.427
#      min_z : -4.427
#      extents : 4.427
#   },
#   meshes: {
#      ...
#   }
# - node 3
#   children: {
#     18,19,20,21,22,23,24,25
#   },
#   bounds: {
#      min_x : 0
#      min_y : -4.427
#      min_z : -4.427
#      extents : 4.427
#   },
#   meshes: {
#      ...
#   }

This should adequately give us the necessary spatial data structure. Later, we can query this octree by testing to see if it intersects with our query sphere.

In [ ]:
gltf_export = pygltflib.GLTF2()
gltf_export.scenes.append(pygltflib.Scene(nodes=[]))
gltf_export.scene = 0